In [12]:
import pandas as pd
import networkx as nx
import requests

In [14]:
# List: 15 Highly Validated Breast Cancer Genes
SEED_GENES = [
    "BRCA1", "BRCA2", "TP53", "PTEN", "PIK3CA", 
    "EGFR", "ERBB2", "AKT1", "CDH1", "STK11",
    "ATM", "CHEK2", "BARD1", "PALB2", "RAD51C"
]

def map_genes_to_string_ids(genes):
    """Queries the STRING REST API to map Gene Symbols to Ensembl Protein IDs."""
    print("Translating Gene Symbols via STRING API...")
    url = "https://string-db.org/api/json/get_string_ids"
    params = {"identifiers": "\r".join(genes), "species": 9606, "limit": 1}
    
    response = requests.post(url, data=params)
    if response.status_code != 200:
        raise ConnectionError("STRING API Connection Failed.")
        
    data = response.json()
    mapped_ids = [entry['stringId'].replace('9606.', '') for entry in data]
    print(f"Successfully mapped {len(mapped_ids)}/{len(genes)} seed genes.")
    return mapped_ids

def build_labeled_disease_subgraph(file_path, seeds, hop_radius=1):
    """Loads DB, filters confidence, extracts subgraph, and calculates MNC labels."""
    print("Loading and filtering STRING database into RAM...")
    df = pd.read_csv(file_path, sep=' ')
    
    # Filter for high confidence physical interactions (>= 700)
    df_filtered = df[df['combined_score'] >= 700].copy()
    df_filtered['protein1'] = df_filtered['protein1'].str.replace('9606.', '')
    df_filtered['protein2'] = df_filtered['protein2'].str.replace('9606.', '')
    
    print("Building global human network...")
    G_global = nx.from_pandas_edgelist(
        df_filtered, source='protein1', target='protein2', edge_attr='combined_score'
    )
    
    print("Extracting localized Breast Cancer subgraph...")
    valid_seeds = [gene for gene in seeds if gene in G_global.nodes()]
    subgraph_nodes = set(valid_seeds)
    
    # Extract 1-hop radius around seed genes
    for seed in valid_seeds:
        subgraph_nodes.update(list(G_global.neighbors(seed)))
        
    G_disease = G_global.subgraph(subgraph_nodes).copy()
    
    print("Calculating heuristic target labels (MNC)...")
    mnc_labels = {}
    for node in G_disease.nodes():
        neighbors = list(G_disease.neighbors(node))
        if not neighbors:
            mnc_labels[node] = 0
            continue
        local_subgraph = G_disease.subgraph(neighbors)
        components = list(nx.connected_components(local_subgraph))
        mnc_labels[node] = len(max(components, key=len)) if components else 0
        
    # Inject labels into network attributes
    nx.set_node_attributes(G_disease, mnc_labels, 'mnc_score')
    
    return G_disease


In [15]:
# EXECUTE PIPELINE 
STRING_FILE = "data/9606.protein.physical.links.v12.0.txt"
OUTPUT_FILE = "breast_cancer_subgraph.graphml"

try:
    translated_seeds = map_genes_to_string_ids(SEED_GENES)
    G_breast_cancer = build_labeled_disease_subgraph(STRING_FILE, translated_seeds)
    
    # Save the prepared ML dataset to disk
    nx.write_graphml(G_breast_cancer, OUTPUT_FILE)
    
    print("\nPIPELINE COMPLETE.")
    print(f"Total Proteins (Nodes): {len(G_breast_cancer.nodes())}")
    print(f"Total Interactions (Edges): {len(G_breast_cancer.edges())}")
    print(f"Labeled graph saved locally as: {OUTPUT_FILE}")

except FileNotFoundError:
    print(f"ERROR: Could not locate '{STRING_FILE}'. Ensure it is in the active directory.")

Translating Gene Symbols via STRING API...
Successfully mapped 15/15 seed genes.
Loading and filtering STRING database into RAM...
Building global human network...
Extracting localized Breast Cancer subgraph...
Calculating heuristic target labels (MNC)...

PIPELINE COMPLETE.
Total Proteins (Nodes): 813
Total Interactions (Edges): 10280
Labeled graph saved locally as: breast_cancer_subgraph.graphml
